# 3D diffusion measured with the mean square displacement

Light atoms (mass 1) start in a slab in the middle of the box, surrounded by heavy atoms (mass 100). The mean square displacement of each group is computed with `compute msd`, and the diffusion coefficient in three dimensions follows from $\langle r^2 \rangle = 6Dt$.

In [ ]:
# lammps-logfile is served from Atomify's own package index (no network needed).
%pip install -q lammps-logfile pandas

In [ ]:
import glob, os, time
import lammps_logfile
import matplotlib.pyplot as plt

# Atomify stores every run of this project in runs/<run-name>/ next to this
# notebook (input snapshot, log.lammps, dumps). Pick the newest run here;
# use logs[0], logs[1], ... to look at older ones.
logs = sorted(glob.glob("runs/*/log.lammps"))
if not logs:
    raise RuntimeError("No runs yet: press Run in Atomify, wait for it to finish, then re-run this cell.")
print("Runs found:", *logs, sep="\n  ")

for attempt in range(5):
    try:
        log = lammps_logfile.File(logs[-1])
        break
    except FileNotFoundError:
        # Atomify may still be copying the finished run into the project.
        time.sleep(1)
        os.listdir(os.path.dirname(logs[-1]))
else:
    raise RuntimeError(f"{logs[-1]} is not readable yet: wait for the run to finish, then re-run this cell.")
print("Log keywords:", log.get_keywords())

In [ ]:
x = log.get("Time")
y = log.get("c_msd_light[4]")

plt.figure(figsize=(6, 6))
plt.subplot(221)
plt.plot(x, y)
plt.xlabel("$t$")
plt.ylabel("$<r^2_{light}>$")

plt.subplot(222)
for i in range(log.get_num_partial_logs()):
    x = log.get("Time", run_num=i)
    y = log.get("c_msd_heavy[4]", run_num=i)
    plt.plot(x, y)
    plt.xlabel("$t$")
    plt.ylabel("$<r^2_{heavy}>$")

plt.subplot(223)
x = log.get("Time")
y = log.get("Temp")
plt.plot(x, y)
plt.xlabel("$t$")
plt.ylabel("$T$")

plt.tight_layout()
plt.show()

The diffusion coefficient is the slope of the MSD divided by 6 (three dimensions). Fit the late, linear part:

In [ ]:
import numpy as np
t = log.get("Time")
sel = t > 0.4 * t.max()          # fit the late, linear part of the MSD
D = {}
for name in ("light", "heavy"):
    msd = log.get(f"c_msd_{name}[4]")
    D[name] = np.polyfit(t[sel], msd[sel], 1)[0] / 6      # 3D: <r^2> = 6 D t
    print(f"D_{name} = {D[name]:.4f}")
print(f"ratio D_light / D_heavy = {D['light'] / D['heavy']:.2f}")